# 01.1 - Arrays & Strings

## What is an array?
A block of **contiguous memory** holding elements one after another. Because elements sit side by side, the computer can jump to element `i` with simple arithmetic: `address = start + i × size` → **O(1) access**.

```
index:   0    1    2    3    4
       +----+----+----+----+----+
value: | 10 | 20 | 30 | 40 | 50 |
       +----+----+----+----+----+
```

Python's `list` is a **dynamic array**: it grows automatically (it over-allocates, so `append` is amortised O(1)).
A **string** is an immutable array of characters.

## Where is it used?
| Real system | How arrays are used |
|---|---|
| Images | A photo is a 2D array (matrix) of pixels |
| NumPy / pandas / ML | Everything is arrays and matrices |
| Spreadsheets, game boards | 2D grids |
| Audio | Array of samples (44,100 per second) |
| Databases | Rows stored in pages (arrays of bytes) |
| Any list on screen | Feeds, search results, playlists |

## Operations & complexity
| Operation | Time |
|---|---|
| Access `a[i]` | O(1) |
| Update `a[i] = x` | O(1) |
| Append at end | O(1) amortised |
| Insert / delete at front or middle | **O(n)** (elements shift) |
| Search unsorted | O(n) |
| Search sorted (binary search) | O(log n) |

## 1. Build a dynamic array from scratch
This is roughly what Python's `list` does internally.

In [1]:
class DynamicArray:
    def __init__(self):
        self.capacity = 1
        self.size = 0
        self.data = [None] * self.capacity

    def __len__(self):
        return self.size

    def __getitem__(self, i):
        if not 0 <= i < self.size:
            raise IndexError("index out of range")
        return self.data[i]

    def append(self, value):              # amortised O(1)
        if self.size == self.capacity:
            self._resize(2 * self.capacity)
        self.data[self.size] = value
        self.size += 1

    def insert(self, index, value):       # O(n): shift right
        if self.size == self.capacity:
            self._resize(2 * self.capacity)
        for i in range(self.size, index, -1):
            self.data[i] = self.data[i - 1]
        self.data[index] = value
        self.size += 1

    def remove_at(self, index):           # O(n): shift left
        value = self.data[index]
        for i in range(index, self.size - 1):
            self.data[i] = self.data[i + 1]
        self.size -= 1
        self.data[self.size] = None
        return value

    def _resize(self, new_capacity):      # O(n), but rare
        new_data = [None] * new_capacity
        for i in range(self.size):
            new_data[i] = self.data[i]
        self.data = new_data
        self.capacity = new_capacity

    def __repr__(self):
        return f"DynamicArray({[self.data[i] for i in range(self.size)]}, capacity={self.capacity})"

arr = DynamicArray()
for x in [10, 20, 30, 40, 50]:
    arr.append(x)
print(arr)
arr.insert(0, 5)
print(arr)
print("removed", arr.remove_at(2), "->", arr)
assert len(arr) == 5 and arr[0] == 5

DynamicArray([10, 20, 30, 40, 50], capacity=8)
DynamicArray([5, 10, 20, 30, 40, 50], capacity=8)
removed 20 -> DynamicArray([5, 10, 30, 40, 50], capacity=8)


## 2. Strings essentials
- Immutable → `s[0] = 'x'` fails; build a list and `"".join(...)`.
- Character codes: `ord('a') = 97`, `chr(97) = 'a'`.
- 26-letter count array: `count[ord(c) - ord('a')] += 1`.

In [2]:
s = "Data Structures"
print(s[::-1], s.lower(), s.split(), "-".join(["a", "b"]))
print([ord(c) - ord('a') for c in "abcz"])

# Building a string: O(n) with join vs O(n^2) with += in a loop (for big n)
pieces = [str(i) for i in range(10)]
print("".join(pieces))

serutcurtS ataD data structures ['Data', 'Structures'] a-b
[0, 1, 2, 25]
0123456789


## 3. Technique: Prefix Sums
Precompute `prefix[i] = sum of first i elements`. Then the sum of any range `[l, r]` = `prefix[r+1] - prefix[l]` in **O(1)**.

Used in: analytics dashboards (sales between two dates), image processing (summed-area tables), range queries.

In [3]:
nums = [3, 1, 4, 1, 5, 9, 2]
prefix = [0]
for x in nums:
    prefix.append(prefix[-1] + x)
print("prefix:", prefix)

def range_sum(l, r):        # inclusive
    return prefix[r + 1] - prefix[l]

assert range_sum(0, 2) == 8      # 3+1+4
assert range_sum(2, 5) == 19     # 4+1+5+9
print("range sums ")

prefix: [0, 3, 4, 8, 9, 14, 23, 25]
range sums 


---
## Solved problems

### 1. Running Sum of 1d Array - LC #1480 (Easy)
Return `running[i] = nums[0] + ... + nums[i]`.

In [4]:
def running_sum(nums):
    result, total = [], 0
    for x in nums:
        total += x
        result.append(total)
    return result

# Time O(n), Space O(n) for output
assert running_sum([1, 2, 3, 4]) == [1, 3, 6, 10]
assert running_sum([3, 1, 2, 10, 1]) == [3, 4, 6, 16, 17]
print("Running Sum ")

Running Sum 


### 2. Reverse String in place - LC #344 (Easy)
Reverse a list of characters using O(1) extra memory.

**Idea:** Two pointers from both ends, swap, move inward.

In [5]:
def reverse_string(s):
    l, r = 0, len(s) - 1
    while l < r:
        s[l], s[r] = s[r], s[l]
        l += 1
        r -= 1

chars = list("hello")
reverse_string(chars)
assert chars == list("olleh")
print("Reverse String ")

Reverse String 


### 3. Longest Common Prefix - LC #14 (Easy)
Longest prefix shared by all strings.

**Idea:** Start with the first word as the prefix; shorten it until every word starts with it.

In [6]:
def longest_common_prefix(strs):
    prefix = strs[0]
    for word in strs[1:]:
        while not word.startswith(prefix):
            prefix = prefix[:-1]
            if not prefix:
                return ""
    return prefix

assert longest_common_prefix(["flower", "flow", "flight"]) == "fl"
assert longest_common_prefix(["dog", "racecar", "car"]) == ""
print("Longest Common Prefix ")

Longest Common Prefix 


### 4. Rotate Array - LC #189 (Medium)
Rotate `nums` to the right by `k` steps, in place.

**Idea:** Reverse the whole array, then reverse the first k, then reverse the rest.

In [7]:
def rotate(nums, k):
    n = len(nums)
    k %= n

    def reverse(l, r):
        while l < r:
            nums[l], nums[r] = nums[r], nums[l]
            l += 1; r -= 1

    reverse(0, n - 1)      # [7,6,5,4,3,2,1]
    reverse(0, k - 1)      # [5,6,7,4,3,2,1]
    reverse(k, n - 1)      # [5,6,7,1,2,3,4]

# Time O(n), Space O(1)
a = [1, 2, 3, 4, 5, 6, 7]; rotate(a, 3); assert a == [5, 6, 7, 1, 2, 3, 4]
b = [-1, -100, 3, 99]; rotate(b, 2); assert b == [3, 99, -1, -100]
print("Rotate Array ")

Rotate Array 


### 5. Subarray Sum Equals K - LC #560 (Medium) Count contiguous subarrays whose sum equals `k` (numbers can be negative).

**Idea:** Prefix sums + hash map. A subarray (i, j] sums to k when `prefix[j] - prefix[i] == k`, i.e. we need how many earlier prefixes equal `prefix[j] - k`.

In [8]:
from collections import defaultdict

def subarray_sum(nums, k):
    count = 0
    prefix = 0
    seen = defaultdict(int)
    seen[0] = 1                      # empty prefix
    for x in nums:
        prefix += x
        count += seen[prefix - k]    # earlier prefixes that make sum k
        seen[prefix] += 1
    return count

# Time O(n), Space O(n)
assert subarray_sum([1, 1, 1], 2) == 2
assert subarray_sum([1, 2, 3], 3) == 2
assert subarray_sum([1, -1, 0], 0) == 3
print("Subarray Sum Equals K ")

Subarray Sum Equals K 


### 6. Rotate Image (matrix) - LC #48 (Medium)
Rotate an n×n matrix 90° clockwise in place.

**Idea:** Rotate 90° clockwise = **transpose** (swap `m[i][j]` and `m[j][i]`) then **reverse each row**. Used in image editors.

In [9]:
def rotate_matrix(matrix):
    n = len(matrix)
    for i in range(n):                   # 1) transpose
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:                   # 2) reverse each row
        row.reverse()

m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate_matrix(m)
assert m == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
print("Rotate Image ")

Rotate Image 


### 7. Spiral Matrix - LC #54 (Medium)
Return all elements of an m×n matrix in spiral order.

**Idea:** Keep four boundaries (top, bottom, left, right); walk one side, shrink that boundary, repeat.

In [10]:
def spiral_order(matrix):
    result = []
    top, bottom = 0, len(matrix) - 1
    left, right = 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            result.append(matrix[top][c])
        top += 1
        for r in range(top, bottom + 1):
            result.append(matrix[r][right])
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                result.append(matrix[bottom][c])
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                result.append(matrix[r][left])
            left += 1
    return result

assert spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral_order([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
print("Spiral Matrix ")

Spiral Matrix 


## More practice on LeetCode
Easy: Remove Duplicates from Sorted Array (#26), Plus One (#66), Merge Sorted Array (#88), Pascal's Triangle (#118)
Medium: Set Matrix Zeroes (#73), Find All Duplicates (#442), Longest Palindromic Substring (#5)

See also `03-Interview Patterns/03.1` for Two Sum, two pointers and sliding window on arrays.

## Key takeaways
- O(1) access by index; inserting/deleting in the middle is O(n).
- Prefix sums turn range-sum queries into O(1).
- Strings are immutable - use lists + `join`.

**Next: `01.2-Hash-Tables-and-Sets.ipynb`**